# Plate 04 · Recogniser Training (CRNN-CTC on device, PARSeq on server)

**On-device recogniser – CRNN:** MobileNetV3-Small feature extractor → BiLSTM (2×128) → CTC.
≈1.5 M params, **≈4 MB int8**, a few ms per crop – meets the < 1.5 s end-to-end budget.

**Server fallback – PARSeq** (permuted autoregressive ViT, Apache-2.0): stronger on blurry / dirty crops;
used only when the phone's confidence < 0.85 (spec section 11.1). Fine-tune with the official `parseq` repo.

Data: real crops (notebook 01) + synthetic (notebook 02), mixed 1:1 at the start, then real-only for the last epochs.

In [ ]:
# --- Setup: make Common_Utils importable (works locally, on Kaggle and on Colab) ---
import sys
from pathlib import Path

# On Kaggle/Colab first clone the repo, e.g.:
# !git clone https://github.com/AhmedADesoky/Car_Services_App.git && %cd Car_Services_App/AI_Models/Plate_Recognition
AI_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(AI_ROOT))

from Common_Utils.Paths import DATA, RUNS, WEIGHTS, model_dir
from Common_Utils.Data_Utils import seed_everything
seed_everything(42)
print("Data:", DATA, "| Runs:", RUNS, "| Weights:", WEIGHTS)

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from Common_Utils.Plate_Normalizer import CHARSET

BLANK = 0
VOCAB = ["<blank>"] + list(CHARSET) + [" "]
CHAR_TO_ID = {c: i for i, c in enumerate(VOCAB)}
device = "cuda" if torch.cuda.is_available() else "cpu"
len(VOCAB)

In [ ]:
class Plate_Crops(Dataset):
    def __init__(self, root, labels_csv, train=True, width=160, height=48):
        self.root, self.df = Path(root), pd.read_csv(labels_csv)
        aug = [transforms.RandomAffine(3, translate=(0.03, 0.05), shear=4), transforms.ColorJitter(0.4, 0.4, 0.3)] if train else []
        self.tf = transforms.Compose(aug + [transforms.Grayscale(), transforms.Resize((height, width)), transforms.ToTensor(),
                                            transforms.Normalize([0.5], [0.5])])
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        image = self.tf(Image.open(self.root / row["file"]).convert("RGB"))
        target = torch.tensor([CHAR_TO_ID[c] for c in str(row["text"]) if c in CHAR_TO_ID], dtype=torch.long)
        return image, target

def collate(batch):
    images, targets = zip(*batch)
    lengths = torch.tensor([len(t) for t in targets])
    return torch.stack(images), torch.cat(targets), lengths

In [ ]:
class CRNN(nn.Module):
    """MobileNetV3-Small backbone (stride 4 in height) + BiLSTM + CTC head."""
    def __init__(self, num_classes=len(VOCAB)):
        super().__init__()
        mb = torchvision.models.mobilenet_v3_small(weights="DEFAULT").features
        mb[0][0] = nn.Conv2d(1, 16, 3, stride=(2, 1), padding=1, bias=False)   # grayscale, keep width
        self.backbone = mb[:9]                                              # -> C=48
        self.pool = nn.AdaptiveAvgPool2d((1, None))
        self.rnn = nn.LSTM(48, 128, num_layers=2, bidirectional=True, batch_first=True, dropout=0.1)
        self.head = nn.Linear(256, num_classes)
    def forward(self, x):                       # x: B x 1 x 48 x 160
        f = self.pool(self.backbone(x)).squeeze(2).permute(0, 2, 1)   # B x T x C
        out, _ = self.rnn(f)
        return self.head(out).log_softmax(-1)  # B x T x K

model = CRNN().to(device)
print(sum(p.numel() for p in model.parameters()) / 1e6, "M params")
print("time steps:", model(torch.zeros(1, 1, 48, 160, device=device)).shape)

In [ ]:
def greedy_decode(log_probs):
    best = log_probs.argmax(-1).cpu().tolist()
    texts, confs = [], []
    probs = log_probs.exp().max(-1).values.cpu()
    for seq, p in zip(best, probs):
        chars, prev = [], BLANK
        for t in seq:
            if t != prev and t != BLANK:
                chars.append(VOCAB[t])
            prev = t
        texts.append("".join(chars).strip())
        confs.append(float(p.mean()))           # simple confidence proxy (min-prob also works well)
    return texts, confs

REC = DATA / "Plates" / "Recognition"
train_ds = Plate_Crops(REC, REC / "labels.csv")   # concat with the synthetic set via torch.utils.data.ConcatDataset
train_dl = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, collate_fn=collate)

In [ ]:
from Common_Utils.Metrics_Utils import exact_match, character_error_rate

ctc = nn.CTCLoss(blank=BLANK, zero_infinity=True)
optim = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
EPOCHS = 60
sched = torch.optim.lr_scheduler.OneCycleLR(optim, max_lr=1e-3, total_steps=EPOCHS * max(1, len(train_dl)))
scaler = torch.amp.GradScaler(enabled=device == "cuda")

for epoch in range(EPOCHS):
    model.train()
    for images, targets, lengths in train_dl:
        images = images.to(device)
        with torch.autocast(device, enabled=device == "cuda"):
            log_probs = model(images)
        T = log_probs.size(1)
        loss = ctc(log_probs.float().permute(1, 0, 2), targets, torch.full((images.size(0),), T, dtype=torch.long), lengths)
        optim.zero_grad(); scaler.scale(loss).backward(); scaler.step(optim); scaler.update(); sched.step()
    print(f"epoch {epoch:03d} loss {loss.item():.4f}")
    torch.save(model.state_dict(), RUNS / "crnn_last.pt")

### Server model: PARSeq fine-tune
```bash
git clone https://github.com/baudm/parseq && cd parseq && pip install -r requirements/core.txt -e .
# Build an LMDB from Data/Plates/Recognition + Synthetic with tools/create_lmdb_dataset.py
./train.py pretrained=parseq-tiny charset_train="<Arabic letters + Arabic-Indic digits + Latin>" \
           data.root_dir=<lmdb_dir> trainer.max_epochs=40 model.lr=2e-4
```
Export the best checkpoint to ONNX (notebook 06) for the FastAPI inference service.